In [17]:
import random

class TabuSearchNQueens:
    def __init__(self, n, max_iterations=10, tabu_tenure=10):
        self.n = n
        self.max_iterations = max_iterations
        self.tabu_tenure = tabu_tenure
        self.tabu_list = []  # Stores moves as (row1, row2) tuples

    def generate_initial_solution(self):
        """Generates a random permutation of columns."""
        solution = list(range(self.n))
        random.shuffle(solution)
        return solution

    def calculate_conflicts(self, solution):
        """
        Calculates the number of pairs of queens attacking each other.
        Since we use a permutation (1 queen per row/col), we only check diagonals.
        """
        conflicts = 0
        n = len(solution)
        for i in range(n):
            for j in range(i + 1, n):
                # Check diagonal attack: |row_diff| == |col_diff|
                if abs(i - j) == abs(solution[i] - solution[j]):
                    conflicts += 1
        return conflicts

    def get_neighbors(self, solution):
        """Generates all possible neighbors by swapping two rows."""
        neighbors = []
        for i in range(self.n):
            for j in range(i + 1, self.n):
                neighbor = solution[:]
                neighbor[i], neighbor[j] = neighbor[j], neighbor[i]
                move = (i, j)
                neighbors.append((neighbor, move))
        return neighbors

    def solve(self):
        current_solution = self.generate_initial_solution()
        best_solution = current_solution[:]
        
        current_conflicts = self.calculate_conflicts(current_solution)
        best_conflicts = current_conflicts
        
        print(f"Initial Conflicts: {best_conflicts}")

        for iteration in range(self.max_iterations):
            if best_conflicts == 0:
                print(f"Solution found at iteration {iteration}!")
                break

            neighbors = self.get_neighbors(current_solution)
            best_neighbor = None
            best_neighbor_conflicts = float('inf')
            best_move = None

            # Iterate through neighbors to find the best allowed move
            for neighbor, move in neighbors:
                conflicts = self.calculate_conflicts(neighbor)
                
                # Aspiration Criterion: Allow tabu move if it improves the global best
                is_tabu = move in self.tabu_list
                if (not is_tabu) or (conflicts < best_conflicts):
                    if conflicts < best_neighbor_conflicts:
                        best_neighbor = neighbor
                        best_neighbor_conflicts = conflicts
                        best_move = move

            if best_neighbor is None:
                print("No valid moves found (stuck in local optima).")
                break

            # Move to the best neighbor found
            current_solution = best_neighbor
            current_conflicts = best_neighbor_conflicts

            # Update global best if needed
            if current_conflicts < best_conflicts:
                best_solution = current_solution[:]
                best_conflicts = current_conflicts

            # Update Tabu List
            self.tabu_list.append(best_move)
            if len(self.tabu_list) > self.tabu_tenure:
                self.tabu_list.pop(0)  # Remove oldest move

        return best_solution, best_conflicts

    def print_board(self, solution):
        for row in range(self.n):
            line = ""
            for col in range(self.n):
                if solution[row] == col:
                    line += "Q "
                else:
                    line += ". "
            print(line)

if __name__ == "__main__":
    N = 8  # Size of the board (8-Queens)
    solver = TabuSearchNQueens(n=N, max_iterations=500, tabu_tenure=5)
    
    solution, conflicts = solver.solve()
    
    print("\nFinal Solution (Column positions for each row):", solution)
    print("Final Conflicts:", conflicts)
    print("\nBoard Visualization:")
    solver.print_board(solution)

Initial Conflicts: 2
Solution found at iteration 1!

Final Solution (Column positions for each row): [4, 1, 7, 0, 3, 6, 2, 5]
Final Conflicts: 0

Board Visualization:
. . . . Q . . . 
. Q . . . . . . 
. . . . . . . Q 
Q . . . . . . . 
. . . Q . . . . 
. . . . . . Q . 
. . Q . . . . . 
. . . . . Q . . 
